## 라쿠텐 단일 리뷰데이터 추출 코드

In [ ]:
import requests
import json
import time
import pandas as pd

def get_rakuten_all_reviews(shop_id, item_id):
    all_reviews = []
    url = "https://web-gateway.rakuten.co.jp/review/itemshopreviewlist/get/v1"
    
    # ⚠️ 쿠키 인코딩 에러 방지 처리
    raw_cookie = r"_ra=1773145336745|9810936d-5cda-4630-8e49-c98379684815; Rp=dc4c4487b0d3c4dce21f7d6bdbf70329f03486c8; rcxGlobal=d45a9a4c-4afa-4759-a9ed-229577264fb8; bm_mi=02819FD1874A677A9FC3219D127ECD18~YAAQB9ojF3ykzaGcAQAAMAK01x85bJZA06vzzbG4DIKAXg7mopEgiKN5lzrV/0AErGfSM50oZ5QfIMqbrcROt7DYSkrYw58Cz7vwEO2XkjKMkHGsBRiolwhSHY6BjhqOg8UrHpjdOwbKwGHWbgXrhYVrXbpqnuYxOnS6Rd4tbRsuPIrmZrskES38smu59dSiBK0bGYfeffgKl/2sz7WT7sEbAOAom3GeHhE5vSnLp8Unxda00YTc54V1R0SccenoAvO/8tAjlqf4fcZMyVfgWjmkkXpBvobpj5hyrWagxWfFFgn06n/L/mwrI22pddxKke0Bln2zVz7w9QvyrImNfBfU7zu1YGnnS6HKKDs9fU21qukz3/qjiTlpM3yQIgMrYmSo2aL/dooM6uWJl/I7WeDfGzeHGvHpBy6zNQ==~1; bm_sv=F3073B8724961BB7DCBE5A9516653181~YAAQbIj+eWOAwsucAQAA7Di01x/kkLOsyzmShVkJPcJf1O0TkzKdl7ZL7qgaen5E33IuCROVCPlrVdQr767lxL25yZRnHKhQTLV74swQf/WGDo3WVE2ehk0W7uv6G6PUyNE2N5dQyBl/0tnQF1TIL1GoIY1QVyysRmPhNLu9lndOUVbr8CgHlwjK+8cZfXcWG/rrLNFiEsHCtxhiQUijgpAP4TGE1raLDQ8pAaWOcclxAHnAjyBcQ2TkH+CwcSLNeCnH~1; ak_bmsc=6DA2C442401E92959215D4FC32333198~000000000000000000000000000000~YAAQB9ojF5AVzqGcAQAA0cu01x9fj3L5e+d9RAJ1hoLlH3K8NWtbYVM99UU1cPhBjs0l+RT4L93zSS2KCBZ2cF79c9IkAVFdcgGHbxP50Uf8nlO5I6u+fsByVK6zS4EhUNSCSizuhMPBdf0lHio/+lSN+8w0fTRgZfWwPNbWNRlfyg8SAriAihvo7Qf2txj9kJZGnaHPqgkCZXs11UwhehZ5Ak3MzYtTcojSc91bzqp+zzmmEbyhuYzqW5FWGr6d5k6LF157rv/BKCBjyW2dtk7TbRgJKgQGZz/gPYD8n8mDMroN36ptPxuByW5GOOwTwoCs8tDnUOjgOtm8M7mmdRNKlpJsUSJoOylT7fkQoeVPZkHsLG1j5QOPG60+P7zXsIeB4pddGO/PKsDF8HPAgLpOVb2Dgqz56kyCR4vfPNXZLawqlDXl8+8HXh2AL5KJmQIZMy0XvT/jsyXGkSBk6KO8l/jsyQp1AG570i/fWulONUZGzkQKF86JqOOaoDS7dUrDgp1mqvDR4lwtHSs=; krt_rewrite_uid=f8b15f33-cc3e-45c0-b1be-9a29a4cff1d2; Re=31.1.5.0.0.216348.3-31.1.5.0.0.216348.3; rat_v=05252dcb01d4e6863512b1a3e769b00e711919a"
    safe_cookie = raw_cookie.encode('utf-8').decode('latin-1', 'ignore')

    headers = {
        "authkey": "isrlPcMjUuXCVBUTVh91ZcHEfoI45CmPR",
        "content-type": "application/json; charset=UTF-8",
        "accept": "application/json, text/plain, */*",
        "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36",
        "cookie": safe_cookie,
        "referer": "https://review.rakuten.co.jp/"
    }

    page = 1
    has_next = True

    print(f"🚀 상품 {item_id} 모든 리뷰 수집 시작 (전체 약 2,200여 건)...")

    while has_next:
        # ✅ 중첩 구조 Payload 반영
        payload = {
            "common": {
                "params": {"device": "pc"},
                "include": ["itemReviewList"]
            },
            "features": {
                "itemReviewList": {
                    "params": {
                        "shopId": int(shop_id),
                        "itemId": int(item_id),
                        "sort": "",
                        "page": str(page),
                        "hits": 30,
                        "filter": {"rating": "", "mediaOnly": "false", "ageRange": "", "sex": ""},
                        "includePickupReview": True
                    }
                }
            }
        }

        try:
            response = requests.post(url, headers=headers, json=payload)
            
            # 200 또는 207 코드가 오면 정상
            if response.status_code not in [200, 207]:
                print(f"\n❌ {page}p 중단 (코드:{response.status_code})")
                break

            data = response.json()
            
            # 파싱 경로: body -> itemReviewList -> data
            res_body = data.get("body", {}).get("itemReviewList", {})
            res_data = res_body.get("data", {})
            reviews = res_data.get("reviews", [])
            
            if not reviews:
                print(f"\n✅ {page}페이지 결과 없음. 수집 완료.")
                break
                
            for rev in reviews:
                all_reviews.append({
                    "Page": page,
                    "Nickname": rev.get("nickname"),
                    "Rating": rev.get("rating"),
                    "Body": rev.get("body"),
                    "PostDate": rev.get("postDate"),
                    "Age": f"{rev.get('ageRange', '')}{rev.get('ageSuffix', '')}",
                    "Sex": rev.get("sex"),
                    "Sku": rev.get("skuInfo")
                })
            
            # ✅ 다음 페이지 존재 여부 확인
            has_next = res_data.get("hasNextPage", False)
            
            print(f"🔄 {page}페이지 완료... (누적 {len(all_reviews)}개)", end='\r')
            
            page += 1
            # 라쿠텐 보안을 고려하여 1.5~2초 간격 권장
            time.sleep(1.8)

        except Exception as e:
            print(f"\n❌ 에러 발생: {e}")
            break

    # 최종 저장
    if all_reviews:
        filename = f"rakuten_{shop_id}_{item_id}_ALL.json"
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(all_reviews, f, ensure_ascii=False, indent=4)
        print(f"\n📂 총 {len(all_reviews)}개 데이터 저장 완료! ('{filename}')")
    
    return pd.DataFrame(all_reviews)

# --- 실행 ---
SHOP_ID = "371043"
ITEM_ID = "10000027"
df = get_rakuten_all_reviews(SHOP_ID, ITEM_ID)